In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader,Dataset,ConcatDataset
import numpy as np
import pandas as pd

In [2]:
train_base="../dataset/train/"
val_base="../dataset/val/"
train_paths=[]
val_paths=[]
for i in range(0,8):
    filename=f"ch{i}.csv"
    train_filepath=train_base+"train_"+filename
    val_filepath=val_base+"val_"+filename
    print(train_filepath)
    print(val_filepath)
    train_paths.append(train_filepath)
    val_paths.append(val_filepath)

../dataset/train/train_ch0.csv
../dataset/val/val_ch0.csv
../dataset/train/train_ch1.csv
../dataset/val/val_ch1.csv
../dataset/train/train_ch2.csv
../dataset/val/val_ch2.csv
../dataset/train/train_ch3.csv
../dataset/val/val_ch3.csv
../dataset/train/train_ch4.csv
../dataset/val/val_ch4.csv
../dataset/train/train_ch5.csv
../dataset/val/val_ch5.csv
../dataset/train/train_ch6.csv
../dataset/val/val_ch6.csv
../dataset/train/train_ch7.csv
../dataset/val/val_ch7.csv


In [21]:
train_df=None
mapping={f"ch{i}": i for i in range(8)}
for path in train_paths:
    df=pd.read_csv(path)
    if train_df is None:
        train_df=df
    else:
        train_df=pd.concat([train_df,df], ignore_index=True)

In [22]:
train_df["channel"] = train_df["channel"].map(mapping)

In [23]:
train_df.head()

,time,channel,PRN,Carrier_Doppler_hz,Pseudorange_m,RX_time,TOW,Carrier_phase,EC,LC,PC,PIP,PQP,TCD,CN0,spoofed,batch_id
0,51506,0,0.0,0.0,0.0,174148.08,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0
1,51507,0,0.0,0.0,0.0,174148.10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0
2,51508,0,0.0,0.0,0.0,174148.12,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0
3,51509,0,0.0,0.0,0.0,174148.14,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0
4,51510,0,0.0,0.0,0.0,174148.16,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0


In [24]:
train_df = train_df.sort_values(by=["time"])

In [25]:
train_df.shape

(6485614, 17)

In [26]:
train_df = train_df.drop(columns=["batch_id"])

In [27]:
train_df = train_df.drop_duplicates()
train_df.head()

,time,channel,PRN,Carrier_Doppler_hz,Pseudorange_m,RX_time,TOW,Carrier_phase,EC,LC,PC,PIP,PQP,TCD,CN0,spoofed
549642,1080,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
549643,1081,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
549744,1082,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
549745,1083,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
549646,1084,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [28]:
train_df.duplicated().sum()

np.int64(0)

In [29]:
train_df.shape

(67014, 16)

In [67]:
X_train = train_df.drop(columns=["spoofed","time"])
y_train = train_df["spoofed"]

In [31]:
val_df=None
mapping={f"ch{i}": i for i in range(8)}
for path in val_paths:
    df=pd.read_csv(path)
    if val_df is None:
        val_df=df
    else:
        val_df=pd.concat([val_df,df], ignore_index=True)

In [32]:
val_df["channel"] = val_df["channel"].map(mapping)
val_df = val_df.sort_values(by=["time"])
val_df = val_df.drop(columns=["batch_id"])
val_df = val_df.drop_duplicates()
val_df.head()

,time,channel,PRN,Carrier_Doppler_hz,Pseudorange_m,RX_time,TOW,Carrier_phase,EC,LC,PC,PIP,PQP,TCD,CN0,spoofed
1410364,5753,4,12.0,3547.192757,776507.881414,491659.46,491659.45741,-315014.446987,115688.13,111385.72,126613.38,126302.26,-8870.5859,3336.5520,46.425213,0
1410465,5754,4,12.0,3548.714204,776494.408171,491659.48,491659.47741,-315085.445188,146228.58,144206.77,155627.11,154434.69,-19228.2130,3335.8748,46.359680,0
1410566,5755,4,12.0,3548.071442,776480.827713,491659.50,491659.49741,-315156.443494,129640.84,123306.73,138162.77,-134699.83,-30739.3160,3342.9844,46.387123,0
1410667,5756,4,12.0,3552.256472,776467.140042,491659.52,491659.51741,-315227.459611,115438.05,156718.11,151602.28,-151178.09,11333.0050,3337.4844,46.479645,0
1410368,5757,4,12.0,3549.527039,776453.702536,491659.54,491659.53741,-315298.478736,131395.31,122371.16,138918.64,-134563.92,-34510.0040,3344.0605,46.514050,0


In [33]:
val_df.shape

(29541, 16)

In [35]:
val_df.duplicated().sum()

np.int64(0)

In [36]:
X_val = val_df.drop(columns=["spoofed","time"])
y_val = val_df["spoofed"]

In [37]:
from xgboost import XGBClassifier

In [38]:
from sklearn.metrics import classification_report, confusion_matrix

In [39]:
model = XGBClassifier()
model.fit(X_train, y_train)
y_pred = model.predict(X_val)

print(classification_report(y_val, y_pred))
print(confusion_matrix(y_val, y_pred))

              precision    recall  f1-score   support

           0       0.94      0.95      0.94     14875
           1       0.94      0.94      0.94     14666

    accuracy                           0.94     29541
   macro avg       0.94      0.94      0.94     29541
weighted avg       0.94      0.94      0.94     29541

[[14063   812]
 [  891 13775]]


In [40]:
from sklearn.ensemble import RandomForestClassifier

In [41]:
model = RandomForestClassifier()
model.fit(X_train, y_train)
y_pred = model.predict(X_val)

print(classification_report(y_val, y_pred))
print(confusion_matrix(y_val, y_pred))

              precision    recall  f1-score   support

           0       1.00      0.98      0.99     14875
           1       0.98      1.00      0.99     14666

    accuracy                           0.99     29541
   macro avg       0.99      0.99      0.99     29541
weighted avg       0.99      0.99      0.99     29541

[[14523   352]
 [    0 14666]]


In [43]:
model.feature_importances_.max()

np.float64(0.34479735252171423)

In [44]:
model.feature_importances_.min()

np.float64(8.988626877938921e-06)

In [45]:
model.feature_importances_

array([1.40084414e-02, 3.89942634e-02, 3.39536997e-02, 1.69324662e-01,
       3.44797353e-01, 3.28837247e-01, 1.05884192e-02, 5.52573736e-03,
       3.30466698e-03, 5.84336519e-03, 1.04520917e-05, 8.98862688e-06,
       2.31234733e-02, 2.16792304e-02])

In [46]:
df=pd.read_csv("../dataset/train.csv")
df.shape

/tmp/ipykernel_24352/3185064830.py:1: DtypeWarning: Columns (0: PRN, 1: Carrier_Doppler_hz, 2: Pseudorange_m, 3: RX_time, 4: TOW, 5: Carrier_phase, 6: EC, 7: LC, 8: PC, 9: PIP, 10: PQP, 11: TCD, 12: CN0) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv("../dataset/train.csv")


(891216, 16)

In [47]:
cols=[]
for col in df.columns:
    if df[col].dtype == "object":
        print(col)
        cols.append(col)

df[cols] = df[cols].apply(pd.to_numeric, errors='coerce')

PRN
Carrier_Doppler_hz
Pseudorange_m
RX_time
TOW
Carrier_phase
EC
LC
PC
PIP
PQP
TCD
CN0


In [48]:
df=df.fillna(0)
df.head()

,time,channel,PRN,Carrier_Doppler_hz,Pseudorange_m,RX_time,TOW,Carrier_phase,EC,LC,PC,PIP,PQP,TCD,CN0,spoofed
0,0,ch0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,0,ch1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,0,ch2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,0,ch3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,0,ch4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [49]:
from sklearn.model_selection import train_test_split

In [50]:
df=df.sort_values(by=["time"])

In [51]:
df=df.drop(columns=["time"])

In [70]:
X = df.drop(columns=["spoofed"])
y = df["spoofed"]

In [71]:
X["channel"] = X["channel"].map(mapping)

In [72]:
X_train,X_test,y_train,y_test=train_test_split(X,y, test_size=0.3,stratify=y, random_state=42)

In [73]:
X_train.shape

(623851, 14)

In [74]:
X_test.shape

(267365, 14)

In [75]:
y_test.value_counts()

spoofed
0    229167
1     38198
Name: count, dtype: int64

In [76]:
y_train.value_counts()

spoofed
0    534721
1     89130
Name: count, dtype: int64

In [77]:
model = XGBClassifier()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.97      1.00      0.98    229167
           1       1.00      0.82      0.90     38198

    accuracy                           0.97    267365
   macro avg       0.98      0.91      0.94    267365
weighted avg       0.97      0.97      0.97    267365

[[229042    125]
 [  6974  31224]]


In [78]:
y_val_pred = model.predict(X_val)

print(classification_report(y_val, y_val_pred))
print(confusion_matrix(y_val, y_val_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     14875
           1       1.00      1.00      1.00     14666

    accuracy                           1.00     29541
   macro avg       1.00      1.00      1.00     29541
weighted avg       1.00      1.00      1.00     29541

[[14875     0]
 [    0 14666]]


In [81]:
import optuna
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, roc_auc_score

In [82]:
def objective(trial):
    
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 5),
        "reg_lambda": trial.suggest_float("reg_lambda", 0, 5),
        "eval_metric": "logloss",
        "random_state": 42,
        "n_jobs": -1
    }

    model = XGBClassifier(**params)

    # Train on given training data
    model.fit(X_train, y_train)

    # Predict on FULL X (as you requested)
    y_pred = model.predict(X_test)

    # Optimize F1 score
    f1 = f1_score(y_test, y_pred)

    return f1


# =========================
# RUN OPTUNA
# =========================
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50, n_jobs=-1)

print("Best F1 Score:", study.best_value)
print("Best Params:", study.best_params)


# =========================
# TRAIN FINAL MODEL
# =========================
best_params = study.best_params

model_xgb_new = XGBClassifier(
    **best_params,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

model_xgb_new.fit(X_train, y_train)

# =========================
# FINAL EVALUATION (your format)
# =========================

y_pred_xgb = model_xgb_new.predict(X_test)

print("XGBoost Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("XGBoost Precision:", precision_score(y_test, y_pred_xgb))
print("XGBoost Recall:", recall_score(y_test, y_pred_xgb))
print("XGBoost F1 Score:", f1_score(y_test, y_pred_xgb))
print("XGBoost ROC-AUC:", roc_auc_score(y_test, model_xgb_new.predict_proba(X_test)[:, 1]))

print("\nXGBoost Classification Report:\n")
print(classification_report(y_test, y_pred_xgb))

[I 2026-04-16 18:15:20,596] A new study created in memory with name: no-name-3e1cbb78-1861-4724-9380-2018aa2d152f
[I 2026-04-16 18:15:36,445] Trial 10 finished with value: 0.8972711988619217 and parameters: {'n_estimators': 167, 'max_depth': 3, 'learning_rate': 0.20579781543864245, 'subsample': 0.628882743160903, 'colsample_bytree': 0.6036985887359023, 'gamma': 3.384362554704417, 'min_child_weight': 4, 'reg_alpha': 3.5294255210337777, 'reg_lambda': 3.4113740458293185}. Best is trial 10 with value: 0.8972711988619217.
[I 2026-04-16 18:15:41,342] Trial 8 finished with value: 0.8978617545978747 and parameters: {'n_estimators': 253, 'max_depth': 10, 'learning_rate': 0.21495435076979358, 'subsample': 0.9790982933757908, 'colsample_bytree': 0.7483197076684514, 'gamma': 0.6880502864725851, 'min_child_weight': 8, 'reg_alpha': 3.3946972124142656, 'reg_lambda': 1.0117877653091383}. Best is trial 8 with value: 0.8978617545978747.
[I 2026-04-16 18:15:41,931] Trial 4 finished with value: 0.89755200

Best F1 Score: 0.8978934502839888
Best Params: {'n_estimators': 387, 'max_depth': 10, 'learning_rate': 0.030176264874009352, 'subsample': 0.7319486935517885, 'colsample_bytree': 0.5037261849493797, 'gamma': 0.11457925390938373, 'min_child_weight': 1, 'reg_alpha': 1.3507515088006172, 'reg_lambda': 3.4233400830126595}
XGBoost Accuracy: 0.9734408019000244
XGBoost Precision: 0.9960123775799917
XGBoost Recall: 0.817372637310854
XGBoost F1 Score: 0.8978934502839888
XGBoost ROC-AUC: 0.9741993978563904

XGBoost Classification Report:

              precision    recall  f1-score   support

           0       0.97      1.00      0.98    229167
           1       1.00      0.82      0.90     38198

    accuracy                           0.97    267365
   macro avg       0.98      0.91      0.94    267365
weighted avg       0.97      0.97      0.97    267365



In [83]:
import joblib

In [84]:
joblib.dump(model_xgb_new, "xgb_single_row_model.pkl")

['xgb_single_row_model.pkl']